# 6 WorkFlow Gerencial, futuro=SEP

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente (Google Colab o Jupyter remoto)

**Google Colab:** correr con runtime **Python 3**, luego cambiar a **R**.

**Jupyter remoto:** correr las celdas de setup con kernel **Python** (o saltear la celda de Drive si no está en Colab). El resto del workflow corre en **R**.

Las rutas se adaptan solas: Colab usa `/content/...`, Jupyter remoto usa `~/labo1` (o la variable de entorno `LABO_BASE`).

**Solo Colab:** conectar Google Drive para persistencia de archivos.

**Jupyter remoto:** esta celda detecta el entorno y saltea el mount si no hay `google.colab`.

In [ ]:
# Setup de entorno: Colab monta Drive, Jupyter remoto usa ~/labo1
import os

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/.drive")
    print("Entorno: Google Colab")
except ImportError:
    print("Entorno: Jupyter remoto (sin google.colab)")

if IN_COLAB:
    os.environ["LABO_BASE"] = "/content/buckets/b1"
    os.environ["LABO_DATASETS"] = "/content/datasets"
else:
    labo_base = os.environ.get("LABO_BASE", os.path.expanduser("~/labo1"))
    os.environ["LABO_BASE"] = labo_base
    os.environ["LABO_DATASETS"] = os.path.join(labo_base, "datasets")
    print("LABO_BASE =", labo_base)
    print("LABO_DATASETS =", os.environ["LABO_DATASETS"])

Crea carpetas, configura `kaggle.json` y descarga datasets.

**Colab:** copiar `kaggle.json` a `My Drive/labo1/kaggle/` antes del primer arranque.

**Jupyter remoto:** copiar `kaggle.json` a `~/labo1/kaggle/kaggle.json` (o dejarlo en la raíz del repo; el script lo busca). Opcional: exportar `LABO_BASE=/ruta/a/tu/carpeta`.



In [ ]:
%%bash

set -euo pipefail

if [ -d "/content/.drive" ]; then
  echo "Setup Colab"
  mkdir -p "/content/.drive/My Drive/labo1"
  mkdir -p "/content/buckets"
  ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1
  LABO_BASE="/content/buckets/b1"
  LABO_DATASETS="/content/datasets"
else
  echo "Setup Jupyter remoto"
  LABO_BASE="${LABO_BASE:-$HOME/labo1}"
  LABO_DATASETS="${LABO_DATASETS:-$LABO_BASE/datasets}"
fi

mkdir -p "$LABO_BASE/exp"
mkdir -p "$LABO_BASE/datasets"
mkdir -p "$LABO_DATASETS"
mkdir -p ~/.kaggle

KAGGLE_JSON=""
for candidate in \
  "$LABO_BASE/kaggle/kaggle.json" \
  "$HOME/kaggle.json" \
  "$HOME/Master/LABO1/kaggle.json" \
  "$(pwd)/kaggle.json" \
  "$(pwd)/../../kaggle.json" \
  "$(pwd)/../../../kaggle.json"
do
  if [ -f "$candidate" ]; then
    KAGGLE_JSON="$candidate"
    break
  fi
done

if [ -n "$KAGGLE_JSON" ]; then
  cp "$KAGGLE_JSON" ~/.kaggle/kaggle.json
  chmod 600 ~/.kaggle/kaggle.json
  echo "kaggle.json instalado desde $KAGGLE_JSON"
else
  echo "AVISO: no se encontro kaggle.json"
fi

descargar() {
  carpeta_destino="$LABO_BASE/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if [ ! -f "$carpeta_destino$archivo" ]; then
    wget "$url_origen$archivo" -O "$carpeta_destino$archivo"
  fi

  if [ "$LABO_DATASETS" != "$carpeta_destino" ] && [ ! -f "$LABO_DATASETS/$archivo" ]; then
    cp "$carpeta_destino$archivo" "$LABO_DATASETS/$archivo"
  fi
}

descargar "dataset_pequeno.csv"
descargar "gerencial_competencia_2026.csv.gz"

echo "LABO_BASE=$LABO_BASE"
echo "LABO_DATASETS=$LABO_DATASETS"


## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$experimento <- 6300
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# rutas: Colab /content, Jupyter remoto ~/labo1
if (dir.exists("/content/buckets/b1")) {
  PARAM$paths$base <- "/content/buckets/b1"
  PARAM$paths$datasets <- "/content/datasets"
} else {
  PARAM$paths$base <- path.expand(Sys.getenv("LABO_BASE", "~/labo1"))
  PARAM$paths$datasets <- file.path(PARAM$paths$base, "datasets")
}

PARAM$kaggle$competencia <- "labo-1-ros-2026-manager"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)

PARAM$montecarlo <- list(
  n_semillas = 7,
  semillas = PARAM$semilla_primigenia + 1:7,
  top_configs = length(PARAM$kaggle$cortes),
  min_cor_predicciones = 0.80,
  min_overlap_topk = 0.80
)

PARAM$paths

#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd(file.path(PARAM$paths$base, "exp"))
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd(file.path(PARAM$paths$base, "exp", experimento_folder))

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(file.path(PARAM$paths$datasets, PARAM$dataset))

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]

#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [ ]:
# sin codigo en esta primera version del workflow

#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# ============================================================
# EXPERIMENTO: Trend3 + Trend6 relativo
# Se crean tendencias temporales normalizadas:
# trend3 = (actual - lag3) / (abs(lag3) + 1)
# trend6 = (actual - lag6) / (abs(lag6) + 1)
# ============================================================

dataset[, paste0(cols_lagueables, "_lag3") := shift(.SD, 3, NA, "lag"),
        by = numero_de_cliente,
        .SDcols = cols_lagueables]

dataset[, paste0(cols_lagueables, "_lag6") := shift(.SD, 6, NA, "lag"),
        by = numero_de_cliente,
        .SDcols = cols_lagueables]

for (vcol in cols_lagueables)
{
  dataset[, paste0(vcol, "_trend3") :=
            (get(vcol) - get(paste0(vcol, "_lag3"))) /
            (abs(get(paste0(vcol, "_lag3"))) + 1)]

  dataset[, paste0(vcol, "_trend6") :=
            (get(vcol) - get(paste0(vcol, "_lag6"))) /
            (abs(get(paste0(vcol, "_lag6"))) + 1)]
}

# Se eliminan lags auxiliares para aislar el efecto de trend3/trend6
dataset[, paste0(cols_lagueables, "_lag3") := NULL]
dataset[, paste0(cols_lagueables, "_lag6") := NULL]

gc()


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset reducido de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 202005, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 202005, 202106 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

# se sacan los meses de pandemia (marzo a junio 2020); en gerencial solo existen 202005 y 202006
PARAM$trainingstrategy$training <- c(
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007
)

PARAM$trainingstrategy$training_pct <- 0.3


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

Esta celda tarda en correr interminables 7 minutos en Colab
<br> ya que debe instalar la librería de LightGBM

In [ ]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE,
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  num_iterations= 2048,
  early_stopping_rounds= 200,
  # --- valores default, pisados por el grid ---
  learning_rate    = 0.05,
  feature_fraction = 0.5,
  bagging_fraction = 0.8,
  bagging_freq     = 1,
  lambda_l1        = 0,
  lambda_l2        = 0,
  min_gain_to_split= 0,
  extra_trees      = FALSE,   # ExtraTrees: umbrales de corte al azar (regularizador)
  num_leaves       = 64,
  min_data_in_leaf = 128
)

# columnas que la busqueda optimiza (se propagan a Monte Carlo)
# bagging_fraction=0.8 y bagging_freq=1 quedan FIJOS (validados en experimentos)
PARAM$lgbm$tune_cols <- c(
  "num_leaves", "min_data_in_leaf", "learning_rate",
  "feature_fraction", "lambda_l1", "lambda_l2",
  "min_gain_to_split", "extra_trees"
)


In [ ]:
# Estima AUC en validation para una combinacion de hiperparametros
# Acepta: num_leaves, min_data_in_leaf, learning_rate,
#         feature_fraction, lambda_l2  (ademas de los fijos)

Estimar_AUC_lightgbm <- function(x) {

  param_completo <- modifyList(PARAM$lgbm$param_fijos, as.list(x))

  modelo_train <- lgb.train(
    data   = dtrain,
    valids = list(valid = dvalidate),
    eval   = "auc",
    param  = param_completo,
    verbose= -100
  )

  AUC   <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]
  niter <- modelo_train$best_iter

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", niter,
    " AUC ",   AUC
  )

  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}


##### Seteo de la busqueda aleatoria (segmentada por rondas)

En vez de un grid fijo, cada ronda muestrea `configs_por_ronda` combinaciones aleatorias del espacio de hiperparametros (orientado a generalizacion). Se corren `n_rondas` segmentos; cada segmento evalua AUC, valida el top con Monte Carlo y sube a Kaggle las configs estables.

`bagging_fraction=0.8` y `bagging_freq=1` quedan fijos (validados). Nuevos parametros explorados: `extra_trees` (ExtraTrees) y `lambda_l1`.

In [ ]:
# ============================================================
# BUSQUEDA ALEATORIA LARGA (multi-dia) - configuracion
# ============================================================
PARAM$busqueda <- list(
  configs_por_ronda = 60,                         # "corridas" por segmento (se evaluan por AUC)
  n_rondas          = 32,                          # cantidad de segmentos (~2 dias al ritmo medio estimado)
  top_mc            = length(PARAM$kaggle$cortes), # cuantas pasan a Monte Carlo por segmento (11)
  rondas_por_submit = 4                            # cada cuantas rondas se sube a Kaggle lo estable acumulado
)
# total de configs evaluadas ~= configs_por_ronda * n_rondas
# cada ronda: muestreo -> AUC -> Monte Carlo del top; las estables se acumulan
# y cada rondas_por_submit se suben a Kaggle todas las estables pendientes

# muestreo aleatorio del espacio, orientado a generalizacion (anti-overfit)
muestrear_configs <- function(n, semilla) {
  set.seed(semilla, kind = "L'Ecuyer-CMRG")
  data.table(
    num_leaves        = sample(16:512, n, replace = TRUE),               # cap moderado: arboles gigantes = overfit
    min_data_in_leaf  = sample(100:6000, n, replace = TRUE),             # alto = mas regularizacion
    learning_rate     = round(10^runif(n, log10(0.01), log10(0.1)), 4),  # ~0.01 a ~0.1 (log)
    feature_fraction  = round(runif(n, 0.20, 0.80), 3),                  # submuestreo de columnas
    lambda_l1         = round(10^runif(n, -3, 1.5), 4),                  # ~0.001 a ~30
    lambda_l2         = round(10^runif(n, -3, 1.5), 4),                  # ~0.001 a ~30
    min_gain_to_split = round(10^runif(n, -3, 0.5), 4),                  # ~0.001 a ~3
    extra_trees       = sample(c(TRUE, FALSE), n, replace = TRUE)        # ExtraTrees on/off
  )
}

cat("Espacio de busqueda listo. tune_cols:", paste(PARAM$lgbm$tune_cols, collapse = ", "), "\n")


(Obsoleto) La corrida ahora ocurre dentro del loop de busqueda mas abajo (celda del loop de rondas). Las celdas intermedias quedan como no-op.

In [ ]:
# (obsoleto) El calculo de AUC ahora corre dentro del loop de busqueda (mas abajo).

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
# (obsoleto) Los resultados se guardan por ronda en registro_configs.txt dentro del loop.

In [ ]:
# (obsoleto) La seleccion del top y el Monte Carlo se hacen dentro del loop de busqueda.

#### 6.3.2.3  Validación Monte Carlo

Las 11 mejores configs se validan con múltiples semillas. `tb_estabilidad` tiene **11 filas** con métricas y gaps vs umbrales. Las predicciones promedio van en `lst_pred_promedio` (aparte).

Si ninguna pasa todos los umbrales, el reporte muestra cuán cerca quedó cada una. Producción usa fallback: config con mayor `cor_min`.


In [ ]:
overlap_topk <- function(pred_a, pred_b, clientes, k) {
  top_a <- clientes[order(-pred_a)][1:k]
  top_b <- clientes[order(-pred_b)][1:k]
  length(intersect(top_a, top_b)) / k
}

estabilidad_preds <- function(preds, clientes, cortes) {
  n <- length(preds)
  cors <- c()
  overlaps <- c()

  for (i in 1:(n - 1)) {
    for (j in (i + 1):n) {
      cors <- c(cors, cor(preds[[i]], preds[[j]]))
      for (k in cortes) {
        overlaps <- c(overlaps, overlap_topk(preds[[i]], preds[[j]], clientes, k))
      }
    }
  }

  list(
    cor_min = min(cors),
    cor_mean = mean(cors),
    overlap_min = min(overlaps),
    overlap_mean = mean(overlaps)
  )
}

entrenar_y_predecir <- function(hiper, semilla, dfinal_train, dfuture, campos_buenos) {
  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  param_run <- modifyList(fijos, hiper)
  param_run$seed <- semilla

  modelo <- lgb.train(
    data = dfinal_train,
    param = param_run,
    verbose = -100
  )

  pred <- predict(
    modelo,
    data.matrix(dfuture[, campos_buenos, with = FALSE])
  )

  list(modelo = modelo, pred = pred)
}

# Corre Monte Carlo (n_semillas) sobre un conjunto de configs candidatas.
# Devuelve tb_estabilidad (metricas + flag estable) y las predicciones promedio.
correr_montecarlo <- function(tb_candidatas) {
  resultados_mc <- lapply(1:nrow(tb_candidatas), function(i) {
    hiper <- as.list(tb_candidatas[i, c(PARAM$lgbm$tune_cols, "num_iterations"), with = FALSE])

    cat(format(Sys.time(), "%X"), " MC config", i, "/", nrow(tb_candidatas),
        " rank=", tb_candidatas[i, config_rank],
        " leaves=", hiper$num_leaves,
        " min_data=", hiper$min_data_in_leaf,
        " extra_trees=", hiper$extra_trees, "\n")

    n_sem_total <- length(PARAM$montecarlo$semillas)
    preds <- lapply(seq_len(n_sem_total), function(j) {
      s <- PARAM$montecarlo$semillas[j]
      cat(format(Sys.time(), "%X"), "    config", i, "/", nrow(tb_candidatas),
          "- semilla", j, "/", n_sem_total, "entrenando...\n")
      entrenar_y_predecir(hiper, s, dfinal_train, dfuture, campos_buenos)$pred
    })

    pred_promedio <- Reduce(`+`, preds) / length(preds)
    est <- estabilidad_preds(preds, dfuture$numero_de_cliente, PARAM$kaggle$cortes)

    corte_k <- min(tb_candidatas[i, corte_asignado], length(pred_promedio))
    n_sem <- length(preds)
    overlaps_corte <- c()
    for (si in 1:(n_sem - 1)) {
      for (sj in (si + 1):n_sem) {
        overlaps_corte <- c(overlaps_corte,
          overlap_topk(preds[[si]], preds[[sj]], dfuture$numero_de_cliente, corte_k))
      }
    }
    overlap_corte_min <- min(overlaps_corte)

    um_cor <- PARAM$montecarlo$min_cor_predicciones
    um_ovl <- PARAM$montecarlo$min_overlap_topk
    pasa_cor <- est$cor_min >= um_cor
    pasa_overlap <- est$overlap_min >= um_ovl
    pasa_corte <- overlap_corte_min >= um_ovl

    cat(format(Sys.time(), "%X"), " >> config", i, "/", nrow(tb_candidatas), "LISTA:",
        " cor_min=", round(est$cor_min, 3),
        " overlap_min=", round(est$overlap_min, 3),
        " overlap_corte_min=", round(overlap_corte_min, 3),
        " estable=", ifelse(pasa_cor && pasa_overlap && pasa_corte, "SI", "NO"), "\n")

    list(
      fila = data.table(
        config_rank = tb_candidatas[i, config_rank],
        corte_asignado = tb_candidatas[i, corte_asignado],
        tb_candidatas[i, c(PARAM$lgbm$tune_cols, "num_iterations", "AUC"), with = FALSE],
        cor_min = est$cor_min, cor_mean = est$cor_mean,
        overlap_min = est$overlap_min, overlap_mean = est$overlap_mean,
        overlap_corte_min = overlap_corte_min,
        umbral_cor = um_cor, umbral_overlap = um_ovl,
        pasa_cor = pasa_cor, pasa_overlap = pasa_overlap, pasa_corte = pasa_corte,
        estable = pasa_cor && pasa_overlap && pasa_corte
      ),
      pred_promedio = pred_promedio
    )
  })

  tb_estabilidad <- rbindlist(lapply(resultados_mc, `[[`, "fila"))
  lst_pred_promedio <- lapply(resultados_mc, `[[`, "pred_promedio")
  names(lst_pred_promedio) <- paste0("rank", tb_estabilidad$config_rank)

  list(tb_estabilidad = tb_estabilidad, lst_pred_promedio = lst_pred_promedio)
}

##### Datasets para Monte Carlo


In [ ]:
# se sacan los meses de pandemia (marzo a junio 2020); en gerencial solo existen 202005 y 202006
PARAM$trainingstrategy$final_train <- c( 202107,
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

dfinal_train <- lgb.Dataset(
  data = data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with = FALSE]),
  label = dataset[fold_final_train == TRUE, clase01],
  free_raw_data = TRUE
)

nrow(dfinal_train)

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
nrow(dfuture)

##### Loop de busqueda (rondas): muestreo -> AUC -> Monte Carlo -> submit

Esta es la celda pesada. Corre `n_rondas` segmentos. En cada uno muestrea configs, evalua AUC, valida el top con Monte Carlo (7 semillas) y sube a Kaggle las estables (respetando el limite de 10/dia). Persiste todo en `registro_configs.txt`, `historial_estabilidad.txt` y `submits_log.txt`.


In [ ]:
# ============================================================
# LOOP DE BUSQUEDA SEGMENTADO POR RONDAS
# Cada ronda: muestreo -> AUC -> Monte Carlo del top -> submit de estables.
# Se corre PARAM$busqueda$n_rondas segmentos. Todo se persiste en disco,
# asi que si se corta, al re-ejecutar no repite configs ni pierde el log de submits.
# ============================================================

# --- log de submits (para registro; NO hay limite diario) ---
archivo_submits <- "submits_log.txt"
submits_log <- if (file.exists(archivo_submits)) {
  fread(archivo_submits, colClasses = list(character = c("ts", "archivo")))
} else data.table(ts = character(), archivo = character(), ronda = integer())

# --- dedup de configs ya evaluadas (firma sobre tune_cols) ---
firma_config <- function(dt) {
  do.call(paste, c(dt[, PARAM$lgbm$tune_cols, with = FALSE], sep = "_"))
}
vistas <- character(0)
if (file.exists("registro_configs.txt")) {
  reg <- fread("registro_configs.txt")
  if (all(PARAM$lgbm$tune_cols %in% names(reg))) vistas <- firma_config(reg)
}

# configs estables acumuladas, pendientes de subir en el proximo checkpoint
pendientes <- list()

for (ronda in 1:PARAM$busqueda$n_rondas) {

  cat("\n=========================================================\n")
  cat(format(Sys.time(), "%Y-%m-%d %X"), " RONDA", ronda, "/", PARAM$busqueda$n_rondas, "\n")
  cat("=========================================================\n")

  # 1) muestreo (semilla distinta por ronda => no repite configs)
  semilla_ronda <- PARAM$semilla_primigenia + ronda * 1000L +
    as.integer(as.numeric(Sys.time())) %% 997L
  tb_cfg <- muestrear_configs(PARAM$busqueda$configs_por_ronda, semilla_ronda)
  tb_cfg <- tb_cfg[!firma_config(tb_cfg) %in% vistas]
  if (nrow(tb_cfg) == 0) { cat("Todas las configs ya vistas; siguiente ronda.\n"); next }

  # 2) AUC en validation (rapido: dtrain con undersampling)
  cat(format(Sys.time(), "%X"), " evaluando AUC de", nrow(tb_cfg), "configs...\n")
  tb_cfg[, c("AUC", "num_iterations") := Estimar_AUC_lightgbm(.SD), by = 1:nrow(tb_cfg)]
  vistas <- c(vistas, firma_config(tb_cfg))
  fwrite(tb_cfg, "registro_configs.txt", sep = "\t", append = TRUE)

  # 3) top configs -> Monte Carlo (evaluacion robusta sobre todos los datos)
  setorder(tb_cfg, -AUC)
  tb_candidatas <- tb_cfg[1:min(PARAM$busqueda$top_mc, .N)]
  tb_candidatas[, config_rank := .I]
  tb_candidatas[, corte_asignado := PARAM$kaggle$cortes[config_rank]]

  mc <- correr_montecarlo(tb_candidatas)
  tb_estabilidad <- mc$tb_estabilidad
  lst_pred_promedio <- mc$lst_pred_promedio
  fwrite(cbind(ronda = ronda, tb_estabilidad), "historial_estabilidad.txt",
         sep = "\t", append = TRUE)

  tb_estables <- tb_estabilidad[estable == TRUE][order(-AUC)]
  cat(format(Sys.time(), "%X"), " ronda", ronda, ":", nrow(tb_estables),
      "configs estables de", nrow(tb_estabilidad), "\n")

  # 4) acumular las estables de esta ronda (con su prediccion y corte)
  if (nrow(tb_estables) > 0) {
    for (i in 1:nrow(tb_estables)) {
      config <- tb_estables[i]
      pendientes[[length(pendientes) + 1L]] <- list(
        ronda  = ronda,
        config = config,
        pred   = lst_pred_promedio[[paste0("rank", config$config_rank)]]
      )
    }
  }

  # 5) CHECKPOINT: cada rondas_por_submit (o en la ultima) subo lo estable acumulado
  es_checkpoint <- (ronda %% PARAM$busqueda$rondas_por_submit == 0) ||
                   (ronda == PARAM$busqueda$n_rondas)

  if (es_checkpoint) {
    if (length(pendientes) == 0) {
      cat(">> checkpoint ronda", ronda, ": sin estables acumuladas para subir.\n")
    } else {
      cat(">> checkpoint ronda", ronda, ": subiendo", length(pendientes),
          "configs estables acumuladas.\n")
      dir.create("kaggle", showWarnings = FALSE)
      for (p in pendientes) {
        config <- p$config
        envios <- min(config$corte_asignado, nrow(dfuture))

        tb_submit <- dfuture[, list(numero_de_cliente)]
        tb_submit[, prob := p$pred]
        setorder(tb_submit, -prob)
        tb_submit[, Predicted := 0L]
        tb_submit[1:envios, Predicted := 1L]

        archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento,
          "_r", p$ronda, "_rank", config$config_rank, "_", envios, ".csv")
        fwrite(tb_submit[, list(numero_de_cliente, Predicted)], file = archivo_kaggle, sep = ",")

        mensaje <- paste0("-m 'ronda=", p$ronda, " rank=", config$config_rank,
          " envios=", envios,
          " estable=SI mc=", PARAM$montecarlo$n_semillas,
          " AUC=", round(config$AUC, 5),
          " cor_min=", round(config$cor_min, 4),
          " overlap_min=", round(config$overlap_min, 4),
          " overlap_corte_min=", round(config$overlap_corte_min, 4),
          " et=", config$extra_trees, "'")
        linea <- paste("kaggle competitions submit",
          paste("-c", PARAM$kaggle$competencia), paste("-f", archivo_kaggle), mensaje)
        salida <- system(linea, intern = TRUE)
        cat(salida, "\n")

        submits_log <- rbind(submits_log,
          data.table(ts = format(Sys.time()), archivo = archivo_kaggle, ronda = p$ronda))
        fwrite(submits_log, archivo_submits, sep = "\t")
        Sys.sleep(30)
      }
      pendientes <- list()  # limpio lo ya subido
    }
  }

  gc()
}

cat("\n### Busqueda finalizada.", PARAM$busqueda$n_rondas, "rondas completadas.\n")
cat("Total configs evaluadas (registro_configs.txt):", length(vistas), "\n")
cat("Total submits realizados:", nrow(submits_log), "\n")

In [ ]:
# Resumen acumulado de todas las rondas (leido del historial persistido).
if (file.exists("historial_estabilidad.txt")) {
  hist_mc <- fread("historial_estabilidad.txt")
  cat("=== Historial de estabilidad (todas las rondas) ===\n")
  print(hist_mc[order(-AUC)][1:min(20, .N),
    .(ronda, config_rank, AUC, cor_min, overlap_min, overlap_corte_min, estable)])
  cat("\nTotal configs con Monte Carlo:", nrow(hist_mc),
      "| estables:", sum(hist_mc$estable), "\n")
} else {
  cat("Aun no hay historial_estabilidad.txt (corra el loop de busqueda).\n")
}

### 6.3.3 Produccion

#### Produccion, Scoring y Kaggle Submit

Solo las configuraciones que pasaron Monte Carlo generan modelos, predicciones y submits.
Cada config estable usa su corte asignado (rank k → cortes[k]).

##### Modelos y predicciones por config estable


In [ ]:
# (obsoleto) Produccion y submit ahora ocurren dentro del loop de busqueda (por ronda).

In [ ]:
# (obsoleto) Ver loop de busqueda.

In [ ]:
# (obsoleto) Ver loop de busqueda.

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle **solo para configs estables**.

Cada config estable sube un solo archivo con su corte asignado (rank 1 → 800, rank 2 → 850, ...).


In [ ]:
# (obsoleto) El submit a Kaggle (throttleado a 10/dia) ocurre dentro del loop de busqueda.

In [ ]:
if (!require("yaml")) install.packages("yaml")
require("yaml")

PARAM$out$busqueda$n_rondas <- if (exists("ronda")) ronda else 0
PARAM$out$busqueda$configs_evaluadas <- if (exists("vistas")) length(vistas) else 0
if (file.exists("historial_estabilidad.txt")) {
  hist_mc <- fread("historial_estabilidad.txt")
  PARAM$out$busqueda$configs_montecarlo <- nrow(hist_mc)
  PARAM$out$busqueda$configs_estables <- sum(hist_mc$estable)
}
if (exists("submits_log")) {
  PARAM$out$busqueda$submits_totales <- nrow(submits_log)
}

write_yaml(PARAM, file = "PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")